# Referable DR: Colab training

Runs the pipeline on the full dataset with a GPU. **Not a medical device.**

Edit only `REPO_URL` (cell 1) and the data source and handle (cell 3). Everything else runs as-is.
Set the runtime to GPU first (Runtime → Change runtime type → T4 GPU).

## 1. GPU check, clone, install

In [ ]:
REPO_URL = "https://github.com/Sammers45/diabetic-retinopathy-mlops.git"

import os, subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No GPU found")
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
%cd /content/repo
%pip install -q -r requirements-colab.txt

## 2. Drive, secrets, environment

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/dr-mlops", exist_ok=True)

# Add a Colab secret named KAGGLE_API_TOKEN (key icon in the left sidebar). It is never printed.
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("Kaggle token loaded from Colab secrets.")
except Exception as exc:  # secret missing or access not granted
    print(f"KAGGLE_API_TOKEN secret not available ({type(exc).__name__}); download will fail.")

os.environ["DR_PROFILE"] = "colab"
os.environ["PYTHONPATH"] = "src"          # for the `!python -m dr...` subprocesses
import sys
sys.path.insert(0, "/content/repo/src")   # for `from dr import ...` in this notebook
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
# Optional: use a remote MLflow server instead of the sqlite store on Drive.
# os.environ["MLFLOW_TRACKING_URI"] = "https://your-mlflow-server"

!python -m dr.config | head -n 40

## 3. Download data

In [ ]:
# Edit these two if you want a different source (see data/README.md).
DATA_SOURCE = "kaggle_dataset"            # kaggle_dataset | kaggle_competition | local
KAGGLE_DATASET = "tanlikesmath/diabetic-retinopathy-resized"

!python -m dr.download --set data.source={DATA_SOURCE} --set data.kaggle_dataset={KAGGLE_DATASET}

## 4. Ingest, validate, split, preprocess
Writes the manifest, the patient-level split and the PNG cache (384 px).

In [ ]:
!python -m dr.data

import json
report = json.load(open("/content/data/processed/data_quality.json"))
print(json.dumps({k: report[k] for k in ["n_images", "n_patients", "referable_share", "grade_counts", "quality_counts", "poor_quality_share"]}, indent=1))
split = json.load(open("/content/data/processed/split_summary.json"))
print(json.dumps({"splits": split["splits"], "leakage": split["leakage"]}, indent=1))

## 5. Train, evaluate, register
EfficientNet-B0 on the full data at 384 px with mixed precision. The threshold is chosen on validation (≥ 90% sensitivity), the test set is scored once, and the model is registered as `dr-referable`.

In [ ]:
!python -m dr.train --run-name effb0-colab

import mlflow
from IPython.display import Image, display
from dr.config import load_config
from dr.tracking import setup_mlflow
setup_mlflow(load_config(), "dr-training")
run = mlflow.search_runs(experiment_names=["dr-training"], order_by=["start_time DESC"], max_results=1).iloc[0]
m = lambda k: run.get(f"metrics.{k}")
print(f"run {run.run_id} | registered v{run.get('tags.registered_version')} @{run.get('tags.registered_alias')}")
print(f"threshold {m('threshold'):.4f} | train minutes {m('train_seconds') / 60:.1f}")
for k in ["roc_auc", "pr_auc", "sensitivity", "specificity", "ppv", "npv", "brier"]:
    print(f"TEST {k:<12} {m('test_' + k):.3f}  [{m('test_' + k + '_ci_low'):.3f}, {m('test_' + k + '_ci_high'):.3f}]")
for name in ["roc_test.png", "calibration_test.png", "confusion_test.png"]:
    display(Image(mlflow.artifacts.download_artifacts(run_id=run.run_id, artifact_path=f"evaluation/{name}")))

## 6. Monitoring replay *(Phase 7)*
## 7. Viewing runs

These cells are added as each phase lands.